# 01 — Data Preparation for Whole-Gland Segmentation

## Objective

This notebook prepares the PI-CAI T2-weighted MRI data for
prostate whole-gland segmentation.

The workflow includes:

1. Dataset and label inspection
2. Image and mask geometry validation
3. Selection of prostate-containing slices with contextual slices
4. T2W intensity normalization
5. Image and mask resizing
6. Generation of the processed 2D dataset
7. Patient-level train, validation, and test split

The processed dataset generated in this notebook is used as input
for the U-Net segmentation model.

## 1. Environment Setup

The project root is added to the Python path so that modules from
the `src` package can be imported from the notebook.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 2. Imports and Data Paths

This section imports the project utilities required for dataset
inspection, preprocessing, visualization, and split generation.

All paths are defined centrally in `src/data/paths.py` to improve
reproducibility and avoid hard-coded directories inside the notebook.

In [ ]:
# Python libraries
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torch
import torch.optim as optim
import importlib

In [ ]:
# Reload imports from the project
import src.data.paths
importlib.reload(src.data.paths)

import src.data.loader
importlib.reload(src.data.loader)

import src.data.mapping
importlib.reload(src.data.mapping)

import src.data.splits
importlib.reload(src.data.splits)

import src.preprocessing.slicing
importlib.reload(src.preprocessing.slicing)

import src.preprocessing.preprocessing
importlib.reload(src.preprocessing.preprocessing)

import src.preprocessing.dataset_builder
importlib.reload(src.preprocessing.dataset_builder)

import src.visualization.visualization
importlib.reload(src.visualization.visualization)

In [ ]:
from src.data.paths import (
    PICAI_DIR,
    PICAI_LABELS_DIR,
    CLINICAL_INFORMATION_DIR,
    PROCESSED_DIR,
    SPLITS_DIR,
)

from src.data.mapping import (
    discover_t2w_cases,
    get_case_image_paths,
    find_whole_gland_mask,
)

from src.data.loader import (
    load_medical_image,
    get_image_metadata,
    validate_image_geometry,
)

from src.preprocessing.slicing import (
    get_positive_slices,
    get_positive_pixel_count,
    select_slices_with_context,
)

from src.preprocessing.preprocessing import (
    normalize_t2w,
    resize_image_and_mask,
)

from src.preprocessing.dataset_builder import (
    process_case,
    validate_case,
    process_dataset,
)

from src.data.splits import (
    create_patient_splits,
)

from src.visualization.visualization import (
    plot_slice_with_mask,
    compare_original_and_processed,
)

## 3. Single-Case Inspection

Before processing the complete dataset, a single PI-CAI case is
inspected to verify image loading, whole-gland mask availability,
spatial compatibility, and slice selection.

This step helps confirm that the preprocessing pipeline behaves as
expected before being applied to all cases.

In [ ]:
patient_id = "10000"
study_id = "1000000"

image_paths = get_case_image_paths(
    patient_id=patient_id,
    study_id=study_id,
)

t2w_path = image_paths["t2w"]

mask_path = find_whole_gland_mask(
    patient_id=patient_id,
    study_id=study_id,
)

print("T2W:", t2w_path)
print("Whole-gland mask:", mask_path)

### 3.1 Image and Mask Loading

The T2W volume and its corresponding whole-gland segmentation mask
are loaded using SimpleITK.

In addition to the array dimensions, the spatial metadata are inspected
because medical images are defined in physical space by properties such
as spacing, origin, and direction.

In [ ]:
t2w_image, t2w_array = load_medical_image(t2w_path)
mask_image, mask_array = load_medical_image(mask_path)

t2w_metadata = get_image_metadata(t2w_image)
mask_metadata = get_image_metadata(mask_image)

print("T2W")
print("Array shape:", t2w_array.shape)
print("Size:", t2w_metadata["size"])
print("Spacing:", t2w_metadata["spacing"])
print("Origin:", t2w_metadata["origin"])
print("Direction:", t2w_metadata["direction"])

print("\nWhole-gland mask")
print("Array shape:", mask_array.shape)
print("Size:", mask_metadata["size"])
print("Spacing:", mask_metadata["spacing"])
print("Origin:", mask_metadata["origin"])
print("Direction:", mask_metadata["direction"])

In [ ]:
same_shape = t2w_array.shape == mask_array.shape

same_geometry = validate_image_geometry(
    t2w_image,
    mask_image,
)

print("Same array shape:", same_shape)
print("Compatible image geometry:", same_geometry)

### 3.2 Whole-Gland Slice Inspection

The whole-gland mask is inspected along the axial dimension to identify
the slices containing prostate tissue.

A small number of contextual slices before and after the prostate region
is also selected. These contextual slices provide examples of adjacent
anatomy while limiting the number of completely irrelevant slices.

In [ ]:
positive_slices = get_positive_slices(mask_array)

selected_slices = select_slices_with_context(
    mask_array,
    context=2,
)

print("Positive slices:", positive_slices)
print("Number of positive slices:", len(positive_slices))

print("\nSelected slices with context:", selected_slices)
print("Number of selected slices:", len(selected_slices))

### 3.3 Visual Inspection of the Whole-Gland Mask

A representative axial slice is visualized with the whole-gland
segmentation mask overlaid on the T2W image.

This qualitative inspection is used to confirm that the mask is
spatially aligned with the prostate anatomy before preprocessing.

In [ ]:
middle_index = len(positive_slices) // 2
slice_index = positive_slices[middle_index]

print("Visualized slice:", slice_index)

plot_slice_with_mask(
    t2w_array,
    mask_array,
    slice_index,
)

### 3.4 Preprocessing Inspection

Before generating the complete 2D dataset, the preprocessing operations
are applied to a representative slice.

The T2W image is normalized using the non-zero intensity values and
resized to 256 × 256 pixels. The corresponding whole-gland mask is
resized using nearest-neighbor interpolation to preserve its binary labels.

The original and processed images are visually compared to verify that
the prostate anatomy and mask alignment are preserved.

In [ ]:
original_image = t2w_array[slice_index]
original_mask = mask_array[slice_index]

normalized_image = normalize_t2w(
    original_image,
)

processed_image, processed_mask = resize_image_and_mask(
    normalized_image,
    original_mask,
    size=(256, 256),
)

print("Original image shape:", original_image.shape)
print("Original mask shape:", original_mask.shape)

print("\nProcessed image shape:", processed_image.shape)
print("Processed mask shape:", processed_mask.shape)

print("\nProcessed image dtype:", processed_image.dtype)
print("Processed mask dtype:", processed_mask.dtype)

In [ ]:
compare_original_and_processed(
    original_image,
    original_mask,
    processed_image,
    processed_mask,
)

## 4. Dataset Discovery and Validation

After validating the preprocessing pipeline on a single case, the
complete PI-CAI dataset is inspected.

This section identifies all available T2W cases and verifies whether
a corresponding whole-gland segmentation mask is available for each
case before full dataset processing.

In [ ]:
cases = discover_t2w_cases()

print("Total T2W cases:", len(cases))
print("\nFirst 5 cases:")

for patient_id, study_id in cases[:5]:
    print(f"{patient_id}_{study_id}")

### 4.1 Whole-Gland Mask Availability

Each T2W case is checked for the presence of a corresponding whole-gland
segmentation mask.

Cases without a segmentation mask would not be eligible for supervised
U-Net training and therefore need to be identified before preprocessing.

In [ ]:
cases_with_mask = []
cases_without_mask = []

for patient_id, study_id in cases:
    mask_path = find_whole_gland_mask(
        patient_id=patient_id,
        study_id=study_id,
    )

    if mask_path is not None:
        cases_with_mask.append(
            (patient_id, study_id)
        )
    else:
        cases_without_mask.append(
            (patient_id, study_id)
        )

print("Total cases:", len(cases))
print("Cases with whole-gland mask:", len(cases_with_mask))
print("Cases without whole-gland mask:", len(cases_without_mask))

### 4.2 Case Validation

All discovered cases are validated before dataset generation.

The validation step checks whether each T2W image and its corresponding
whole-gland mask can be loaded successfully, whether their array
dimensions match, and whether their spatial metadata are compatible.

Spatial compatibility is evaluated using image spacing, origin, and
direction with a tolerance of `1e-3`, which accounts for small numerical
differences in metadata representation between image formats.

This step helps detect corrupted files, missing data, or incompatible
image-mask pairs before preprocessing the complete dataset.

In [ ]:
validation_results = []

for index, (patient_id, study_id) in enumerate(cases, start=1):
    result = validate_case(
        patient_id=patient_id,
        study_id=study_id,
    )

    validation_results.append(result)

    if index % 100 == 0:
        print(f"Validated {index}/{len(cases)} cases")

In [ ]:
valid_cases = [
    result
    for result in validation_results
    if result["valid"]
]

invalid_cases = [
    result
    for result in validation_results
    if not result["valid"]
]

print("Total cases:", len(validation_results))
print("Valid cases:", len(valid_cases))
print("Invalid cases:", len(invalid_cases))

## 5. Processed Dataset Generation

After validating all T2W images and whole-gland masks, the complete
2D segmentation dataset is generated.

For each case, axial slices containing the prostate are selected together
with two contextual slices before and after the prostate region.

Each T2W slice is normalized and resized to 256 × 256 pixels. The
corresponding whole-gland mask is resized using nearest-neighbor
interpolation and preserved as a binary mask.

The processed image-mask pairs are stored as NumPy arrays and will be
used as input for the U-Net segmentation model.

In [ ]:
processed_dataset_dir = (
    PROCESSED_DIR
    / "whole_gland_2d"
)

print(
    "Processed dataset directory:",
    processed_dataset_dir,
)

### 5.1 Complete Dataset Processing

The complete validated dataset can be processed using the preprocessing
pipeline implemented in `src.preprocessing.dataset_builder`.

For each PI-CAI case, the pipeline selects the prostate-containing
axial slices together with two contextual slices, normalizes the T2W
intensities, resizes the images and masks to 256 × 256 pixels, and
stores the resulting image-mask pairs as NumPy arrays.

The processing step is shown below for reproducibility. Since the
processed dataset has already been generated, this cell does not need
to be executed again unless the preprocessing configuration changes.

In [ ]:
processing_summary = process_dataset(
    cases=cases,
    output_dir=processed_dataset_dir,
    context=2,
    image_size=(256, 256),
)

print(processing_summary)

### 5.2 Processed Dataset Verification

The generated dataset contains 28,110 image-mask pairs with one-to-one
correspondence between files.

All processed samples have a spatial resolution of 256 × 256 pixels.
T2W images are stored as `float32` arrays and whole-gland masks as
binary `uint8` arrays containing values 0 and 1.

These checks confirm that the processed dataset is structurally
consistent and ready for patient-level dataset partitioning.

In [ ]:
images_dir = processed_dataset_dir / "images"
masks_dir = processed_dataset_dir / "masks"

image_files = sorted(
    images_dir.glob("*.npy")
)

mask_files = sorted(
    masks_dir.glob("*.npy")
)

print(
    "Processed images:",
    len(image_files),
)

print(
    "Processed masks:",
    len(mask_files),
)

print(
    "Image-mask count match:",
    len(image_files) == len(mask_files),
)

In [ ]:
image_names = {
    path.name
    for path in image_files
}

mask_names = {
    path.name
    for path in mask_files
}

images_without_masks = (
    image_names - mask_names
)

masks_without_images = (
    mask_names - image_names
)

print(
    "Images without masks:",
    len(images_without_masks),
)

print(
    "Masks without images:",
    len(masks_without_images),
)

In [ ]:
import numpy as np

sample_image = np.load(
    image_files[0]
)

sample_mask = np.load(
    mask_files[0]
)

print(
    "Image shape:",
    sample_image.shape,
)

print(
    "Mask shape:",
    sample_mask.shape,
)

print(
    "Image dtype:",
    sample_image.dtype,
)

print(
    "Mask dtype:",
    sample_mask.dtype,
)

print(
    "Mask values:",
    np.unique(sample_mask),
)

In [ ]:
positive_mask_index = next(
    index
    for index, mask_path in enumerate(mask_files)
    if np.any(np.load(mask_path) > 0)
)

positive_image = np.load(
    image_files[positive_mask_index]
)

positive_mask = np.load(
    mask_files[positive_mask_index]
)

print(
    "Positive sample:",
    image_files[positive_mask_index].name,
)

print(
    "Image shape:",
    positive_image.shape,
)

print(
    "Mask shape:",
    positive_mask.shape,
)

print(
    "Mask values:",
    np.unique(positive_mask),
)

print(
    "Positive pixels:",
    int(positive_mask.sum()),
)

### 5.3 Dataset Preparation Summary

The preprocessing pipeline produced 28,110 image-mask pairs from the
1,500 validated PI-CAI cases.

All processed samples have a spatial resolution of 256 × 256 pixels.
T2W images are stored as `float32` arrays, while whole-gland masks are
stored as binary `uint8` arrays.

File correspondence and sample integrity were verified before the
dataset partitioning stage.

## 6. Clinical Data and Dataset Splitting

The PI-CAI clinical information is used to define the patient-level
training, validation, and test partitions.

The `case_csPCa` variable indicates whether clinically significant
prostate cancer is present in each case and is used to stratify the
dataset partitions.

Splitting is performed at the case level before associating the
processed 2D slices with each partition. This prevents slices from the
same MRI examination from being distributed across different subsets.

In [ ]:
clinical_data_path = (
    CLINICAL_INFORMATION_DIR
    / "marksheet.csv"
)

clinical_data = pd.read_csv(
    clinical_data_path
)

print(
    "Clinical records:",
    len(clinical_data),
)

print(
    "\nColumns:"
)

print(
    clinical_data.columns.tolist()
)

### 6.1 Clinically Significant Prostate Cancer Distribution

Before creating the dataset partitions, the distribution of the
`case_csPCa` variable is inspected.

This variable is used for stratification so that the proportion of
cases with and without clinically significant prostate cancer is
approximately preserved across the training, validation, and test sets.

In [ ]:
cspca_counts = (
    clinical_data["case_csPCa"]
    .value_counts()
)

cspca_percentages = (
    clinical_data["case_csPCa"]
    .value_counts(normalize=True)
    .mul(100)
)

print("Case distribution:")
print(cspca_counts)

print("\nPercentage distribution:")
print(
    cspca_percentages.round(2)
)

### 6.2 Patient-Level Train, Validation, and Test Split

The dataset is divided into training, validation, and test subsets using
a 70% / 15% / 15% proportion.

The split is performed using the clinical records rather than individual
2D slices. Therefore, all slices belonging to the same MRI case remain
in the same subset, preventing data leakage between training, validation,
and test data.

Stratification based on `case_csPCa` is applied to approximately preserve
the distribution of clinically significant prostate cancer across the
three subsets.

A fixed random state is used to make the partition reproducible.

In [ ]:
split_data = create_patient_splits(
    clinical_data=clinical_data,
    random_state=42,
)

print(
    split_data["split"]
    .value_counts()
)

In [ ]:
split_distribution = pd.crosstab(
    split_data["split"],
    split_data["case_csPCa"],
)

split_distribution["total"] = (
    split_distribution.sum(axis=1)
)

print(split_distribution)

In [ ]:
split_percentages = pd.crosstab(
    split_data["split"],
    split_data["case_csPCa"],
    normalize="index",
).mul(100)

print(
    split_percentages.round(2)
)

### 6.3 Saving the Dataset Split

The patient-level partition is saved as a CSV file so that the same
training, validation, and test subsets can be consistently reused
throughout the segmentation pipeline.

This avoids generating new random partitions in subsequent experiments
and ensures reproducibility.